# 04 - On-Device Training and Continual Learning

`02` and `03` both assumed a model's weights are finished before it ever reaches the
device - train, export, convert, deploy, done. This notebook asks what happens if that
assumption doesn't hold: what if the model keeps learning *after* it's deployed?

`frc_resources/03_roboflow` names a real, recurring cost of the team's current
workflow: "the target class changes every year... a team effectively restarts step 1
of the loop every single season. There is no 'last year's dataset' to lean on." A
tempting shortcut follows naturally from that: instead of retraining from scratch in
Roboflow every season, why not just fine-tune *last* season's model on this season's
game piece? It already knows how to find *something* game-piece-shaped - surely that's
a head start, not a restart.

This notebook builds a small, controlled version of exactly that experiment, measures
what actually happens, and is honest about the result: the shortcut has a real, well
-known failure mode, and the fix for it is a genuine tradeoff, not a free lunch. Nothing
in this notebook is a settled recommendation for this team - it's the background
needed to make that decision for real, whenever it comes up.

## On-Device Training vs. Train-Elsewhere-Deploy-Here

Every notebook before this one - including `02` and `03` - treated training and
deployment as two separate phases happening in two separate places: train somewhere
unconstrained, deploy a finished, frozen set of weights somewhere constrained.
**On-device training** breaks that separation: the model keeps updating its own
weights using data it sees *after* deployment, on the constrained device itself.
**Continual learning** is the general name for the problem this creates: how do you
keep learning from new data over time without undoing what the model already knew?

`01` already explained why this is harder than it sounds on real edge hardware: a
backward pass needs to store gradients, optimizer state, and intermediate activations
that a forward-pass-only inference chip was never built to hold. The accelerators named
throughout this primer (and the specific ones named in `frc_resources/02_limelight`)
are inference silicon - training on the accelerator itself generally isn't on the
table. The realistic version of "on-device" for a team like this would mean training on
a robot's own general controller (roboRIO / Systemcore) or a coprocessor, not the
dedicated accelerator chip - which puts you right back inside the same compute-budget
argument `01` opened with, just for backpropagation instead of inference.

## Imports

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
rng = np.random.default_rng(0)


## Setting Up Two "Seasons"

To keep this fast, controlled, and honest (no downloaded dataset standing in for
whatever the eventual answer "should" be), both tasks below are synthetic 2D
classification problems, standing in for two seasons' worth of game-piece detection.
Both tasks share the same input region - the same rough camera-view feature space - but
disagree on where the decision boundary goes, which is deliberate: it's what makes
this a fair stand-in for "similar-looking scenes, different game piece to find," not an
easy case where the two tasks never even overlap.

In [ ]:
def make_task(rule, n=400, noise=0.15):
    X = rng.uniform(-3, 3, size=(n, 2))
    y = (rule(X) > 0).astype(np.int64)
    X = X + rng.normal(0, noise, X.shape)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


# Task A ("this season's game piece"): separated along x0 + x1
task_a_rule = lambda X: X[:, 0] + X[:, 1]
# Task B ("next season's game piece"): a genuinely different boundary, same input region
task_b_rule = lambda X: X[:, 0] - X[:, 1]

X_a_train, y_a_train = make_task(task_a_rule, n=400)
X_a_test, y_a_test = make_task(task_a_rule, n=200)
X_b_train, y_b_train = make_task(task_b_rule, n=400)
X_b_test, y_b_test = make_task(task_b_rule, n=200)

fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)
for ax, X, y, title in [
    (axes[0], X_a_train, y_a_train, "Task A (this season)"),
    (axes[1], X_b_train, y_b_train, "Task B (next season)"),
]:
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=10, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("feature 0")
axes[0].set_ylabel("feature 1")
fig.suptitle("Same input region, different decision boundary")
fig.tight_layout()
plt.show()


## A Tiny Classifier, Trained on Task A Only

A small network - nothing here is different in kind from `deep_learning_primer`'s
from-scratch classifier, just built with `torch` directly instead of by hand, since
we'll need real gradients for fine-tuning below.

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16), nn.ReLU(),
            nn.Linear(16, 16), nn.ReLU(),
            nn.Linear(16, 2),
        )

    def forward(self, x):
        return self.net(x)


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X).argmax(dim=1)
    return (preds == y).float().mean().item()


def train_full_batch(model, X, y, epochs, lr=0.1):
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
    return model


base_model = TinyClassifier()
train_full_batch(base_model, X_a_train, y_a_train, epochs=300)

print(f"Task A test accuracy: {accuracy(base_model, X_a_test, y_a_test):.3f}")
print(f"Task B test accuracy: {accuracy(base_model, X_b_test, y_b_test):.3f}  (untrained on B - should be near chance, 0.5)")


## Naive Fine-Tuning: Watching Task A Collapse

Now we take a copy of that Task-A-trained model and fine-tune it on Task B data only -
exactly the shortcut motivated in the introduction. We check Task A and Task B accuracy
every 20 epochs so we can watch what happens over the course of fine-tuning, not just
at the end.

In [ ]:
CHECKPOINTS = list(range(0, 201, 20))


def fine_tune_and_track(model, X_finetune, y_finetune, lr=0.1):
    model = copy.deepcopy(model)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    a_accs, b_accs = [], []
    for epoch in range(max(CHECKPOINTS) + 1):
        if epoch in CHECKPOINTS:
            a_accs.append(accuracy(model, X_a_test, y_a_test))
            b_accs.append(accuracy(model, X_b_test, y_b_test))
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(X_finetune), y_finetune)
        loss.backward()
        opt.step()
    return a_accs, b_accs


naive_a_accs, naive_b_accs = fine_tune_and_track(base_model, X_b_train, y_b_train)

print(f"{'epoch':>6}{'Task A acc':>12}{'Task B acc':>12}")
for e, a, b in zip(CHECKPOINTS, naive_a_accs, naive_b_accs):
    print(f"{e:>6}{a:>12.3f}{b:>12.3f}")


**Catastrophic forgetting**, first named by
[McCloskey and Cohen (1989)](https://www.sciencedirect.com/science/chapter/bookseries/abs/pii/S0079742108605368)
and formalized further by
[French (1999)](https://www.cell.com/trends/cognitive-sciences/abstract/S1364-6613(99)01294-2),
is exactly the pattern above: a network trained sequentially on task after task doesn't
gracefully accumulate knowledge, it sharply loses old-task performance as soon as new
-task training begins. Task B accuracy climbs toward the high 0.90s while Task A
accuracy falls toward chance (0.5) - the fine-tuned "improved" detector has, for
practical purposes, forgotten last season's game piece almost entirely, despite having
scored 0.97 on it a few cells ago.

**Why this happens:** nothing in the fine-tuning loop's loss function has any term that
cares about Task A. Every gradient step is computed purely to reduce Task B's loss, and
the optimizer has no way to know - or reason to care - that some of the weight
directions it's moving through were load-bearing for a different decision boundary.
This is sometimes called the **stability-plasticity dilemma**: a network plastic enough
to learn new data quickly is, by the same property, a network that overwrites old
knowledge quickly too.

## A Real Mitigation: Rehearsal

The simplest well-established fix is **rehearsal** (also called *replay*): keep a
small buffer of old-task examples, and mix them back into every fine-tuning batch so
the loss function has *some* term that still cares about Task A. We keep a buffer of
just 100 Task A examples (a quarter of the original 400) and repeat it to match Task
B's batch size, so it carries roughly equal weight in the loss rather than being
drowned out.

In [ ]:
replay_idx = rng.choice(len(X_a_train), size=100, replace=False)
X_replay_base, y_replay_base = X_a_train[replay_idx], y_a_train[replay_idx]
reps = len(X_b_train) // len(X_replay_base)
X_replay = X_replay_base.repeat(reps, 1)
y_replay = y_replay_base.repeat(reps)

X_b_with_replay = torch.cat([X_b_train, X_replay], dim=0)
y_b_with_replay = torch.cat([y_b_train, y_replay], dim=0)

rehearsal_a_accs, rehearsal_b_accs = fine_tune_and_track(
    base_model, X_b_with_replay, y_b_with_replay
)

print(f"{'epoch':>6}{'Task A acc':>12}{'Task B acc':>12}")
for e, a, b in zip(CHECKPOINTS, rehearsal_a_accs, rehearsal_b_accs):
    print(f"{e:>6}{a:>12.3f}{b:>12.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(CHECKPOINTS, naive_a_accs, "o-", color="tab:red", label="Task A acc (naive fine-tune)")
ax.plot(CHECKPOINTS, naive_b_accs, "o--", color="tab:red", alpha=0.5, label="Task B acc (naive fine-tune)")
ax.plot(CHECKPOINTS, rehearsal_a_accs, "o-", color="tab:blue", label="Task A acc (with rehearsal)")
ax.plot(CHECKPOINTS, rehearsal_b_accs, "o--", color="tab:blue", alpha=0.5, label="Task B acc (with rehearsal)")
ax.axhline(0.5, color="gray", linestyle=":", label="chance level")
ax.set_xlabel("fine-tuning epoch")
ax.set_ylabel("test accuracy")
ax.set_title("Naive fine-tuning vs. rehearsal")
ax.legend(loc="center right", fontsize=8)
fig.tight_layout()
plt.show()


Rehearsal is a real, measured improvement, not a free lunch: Task A accuracy stays
much higher (roughly 0.80 rather than collapsing to roughly 0.55), but Task B only
reaches roughly 0.70 rather than naive fine-tuning's roughly 0.96. That gap is the
stability-plasticity dilemma made concrete - protecting old-task performance costs some
of the new task's learning capacity, on purpose, because every replayed old-task
example in a batch is one fewer new-task example's worth of gradient signal. Rehearsal
isn't the only mitigation family that exists - regularization-based approaches like
[Kirkpatrick et al. (2017)'s Elastic Weight Consolidation](https://arxiv.org/abs/1612.00796)
penalize changes to weights that were important for old tasks, instead of replaying old
data directly - but every approach in this space is trading away some new-task
plasticity to buy back old-task stability. None of them make that trade disappear.

## Where This Leaves the Team

Nothing above is a recommendation to start fine-tuning models on the robot next season
- it's the background for a real decision the team hasn't made yet. Three genuinely
different options sit on the table, in roughly increasing order of ambition and risk:

1. **Keep doing what `frc_resources/03_roboflow` already describes** - retrain from
   scratch each season in Roboflow. No forgetting risk, because there's no old model
   being reused; the cost is exactly the "restart every season" cost that motivated
   this notebook in the first place.
2. **If fine-tuning last season's model is tempting for speed, use rehearsal, not naive
   fine-tuning** - keep a small labeled set of last season's game piece and mix it into
   this season's training data. Cheaper than starting fully from scratch, with a known,
   measurable cost to how well the new task is learned.
3. **True on-device continual learning** - updating weights on the robot itself, from
   data collected mid-competition - is the most ambitious option here, and the least
   proven for this team's use case. It's the frontier worth knowing exists, not a plan
   to execute without a lot more testing than one notebook's worth.

## Try It Yourself

1. Instead of fine-tuning every layer on Task B, freeze the first two layers
   (`for p in model.net[0].parameters(): p.requires_grad = False`, and the same for the
   next `Linear` layer) and fine-tune only the last layer. Does forgetting get better or
   worse? Relate this to `01`'s point about training being heavier than inference - a
   frozen-backbone fine-tune is cheaper to compute *and*, as you'll see, does something
   different to Task A's accuracy.
2. Vary the replay buffer size (try 10, 50, and 200 examples instead of 100). Find
   roughly the smallest buffer that still keeps Task A above 0.9.
3. Change `task_b_rule` to something closer to `task_a_rule` (e.g.
   `lambda X: X[:, 0] + 0.9 * X[:, 1]` instead of the orthogonal `X[:, 0] - X[:, 1]`
   used above). Does forgetting get less severe when the two tasks disagree less?

## Resources

- [McCloskey & Cohen (1989): Catastrophic Interference in Connectionist Networks](https://www.sciencedirect.com/science/chapter/bookseries/abs/pii/S0079742108605368) -
  the original paper naming the phenomenon demonstrated above.
- [French (1999): Catastrophic Forgetting in Connectionist Networks](https://www.cell.com/trends/cognitive-sciences/abstract/S1364-6613(99)01294-2) -
  a widely-cited follow-up survey, source of the "stability-plasticity dilemma" framing
  used above.
- [Kirkpatrick et al. (2017): Overcoming Catastrophic Forgetting in Neural Networks](https://arxiv.org/abs/1612.00796) -
  a well-known, freely available paper on Elastic Weight Consolidation, the
  regularization-based mitigation family mentioned as an alternative to rehearsal.
- `frc_resources/03_roboflow` - the real, current FRC workflow this notebook's opening
  question was testing an alternative to.